In [ ]:
import time
import pandas as pd
import os
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score, precision_recall_fscore_support

from utils import *

import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

hola


In [ ]:
BASE_PATH = Path("../data")
EXP_PATH = BASE_PATH / "EXPERIMENT1"
DOMAIN_PATH = EXP_PATH / "domain_adaptation"

TRAIN_PATH = DOMAIN_PATH / "train_public_64"
VAL_PATH   = DOMAIN_PATH / "val_public_16"
TEST_PUBLIC_PATH = DOMAIN_PATH / "test"   # In-domain test
TEST_LOCAL_PATH  = EXP_PATH / "local_data"          # Out-of-domain test
RESULTS_PATH = EXP_PATH / "models"
OUTPUT_PATH = DOMAIN_PATH / "results"


FIGS_DIRECTORY = BASE_PATH / "paper_figs"
os.makedirs(FIGS_DIRECTORY, exist_ok=True)

BATCH_SIZE = 32

IMG_SIZE = (224,224)
model_list = [
    "EffNetB3_CBAM",
    "ResNet101_CBAM",
    "mobilenetv2",
    "efficientnetb0",
    "mobilenetv3large",
    "densenet121",
    "efficientnetb3",
    "resnet50",
    "inceptionv3",
    "resnet101",
    "vgg16",
    "vgg19",
    "efficientnetb5"
]

NUM_CLASSES = 4

In [ ]:
best_model = df_results.loc[df_results["Δ F1"].idxmin()]
print(f"""
MOST ROBUST MODEL (IN TERMS OF Δ F1):
- {best_model['Model']}
- F1 Público: {best_model['F1_Public']:.3f}
- F1 Local:   {best_model['F1_Local']:.3f}
- Δ F1:       {best_model['Δ F1']:.3f}
""")

# Evaluating CNN models upon known domain (public test)

In [ ]:
TEST_PATH = Path(r"../data/dataset/test")
RESULTS_PATH = Path("../models")
CONF_MATRIX_DIR = RESULTS_PATH / "confusion_matrix"

## Model evaluation (base models and CBAM)

In [ ]:
CLASSES = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])
class_indices = {cls: i for i, cls in enumerate(CLASSES)}

model_list = [
    "mobilenetv2", "efficientnetb0", "efficientnetb3", "mobilenetv3large", "densenet121",
    "resnet50", "resnet101", "Inceptionv3", "vgg16", "vgg19", "efficientnetb5"
]

for backbone_name in model_list:
    print(f"\n=== EVALUATING: {backbone_name} ===\n")

    best_model_path = RESULTS_PATH / f"{backbone_name}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model: {backbone_name} not found, skipping.")
        continue

    model = tf.keras.models.load_model(best_model_path, compile=False)

    X_list, y_list = [], []

    for cls in CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists():
            print(f"Absent class: {cls}")
            continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpeg"))

        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)

            img = img.astype(np.float32) / 255.0
            img = preprocess_for_backbone(img, backbone_name) 

            X_list.append(img)
            y_list.append(class_indices[cls])

    if not X_list:
        print(f"Not available images for model {backbone_name}.")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(preds, axis=1)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n=== GLOBAL METRICS FOR {backbone_name} ===")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"F1-score:       {f1:.4f}")
    print(f"Cohen's Kappa:  {kappa:.4f}\n")

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(CLASSES))
    )

    cm_color = cm.astype(np.float32) / cm.sum(axis=1, keepdims=True)
    cm_color = np.nan_to_num(cm_color)

    plt.figure(figsize=(8, 6)) 

    sns.heatmap(
        cm_color,          
        annot=cm,          
        fmt="d",
        cmap="Blues",
        xticklabels=CLASSES,
        yticklabels=CLASSES,
        vmin=0, vmax=1
    )

    plt.xlabel("Predicted")
    plt.ylabel("Real")


    plt.xticks(rotation=45, ha="right")
    plt.tight_layout() 
    out_path = CONF_MATRIX_DIR / f"confusion_matrix_{backbone_name}.png"
    plt.savefig(out_path, dpi=300)


    plt.show()
    plt.close()

    print(f"Confusion matrix saved to: {out_path}")

## Comparison of models CBAM vs base ones.

In [ ]:
experiments = [
    {"name": "efficientnetb0_CBAM", "backbone": "efficientnetb0"},
    {"name": "mobilenetv2_CBAM",    "backbone": "mobilenetv2"},
    {"name": "efficientnetb3_CBAM", "backbone": "efficientnetb3"},
    {"name": "resnet101_CBAM",  "backbone": "resnet101"},
]

In [ ]:
colors = ["#1f77b4", "#ff7f0e"]

def plot_model_metrics(df, model_name, classes, save_path=None):
    sub = df[df["Model"] == model_name]
    base = sub[sub["Variant"]=="base"].iloc[0]
    cbam = sub[sub["Variant"]=="cbam"].iloc[0]

    f1_base = np.array(base["F1_per_class"], dtype=float)
    f1_cbam = np.array(cbam["F1_per_class"], dtype=float)
    prec_base = np.array(base["Precision_per_class"], dtype=float)
    prec_cbam = np.array(cbam["Precision_per_class"], dtype=float)
    rec_base = np.array(base["Recall_per_class"], dtype=float)
    rec_cbam = np.array(cbam["Recall_per_class"], dtype=float)

    x = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(1, 3, figsize=(13,4))
    fig.subplots_adjust(wspace=0.3)

    # --- F1 ---
    ax[0].bar(x - width/2, f1_base, width, label="Base", color=colors[0])
    ax[0].bar(x + width/2, f1_cbam, width, label="CBAM", color=colors[1])
    ax[0].set_xticks(x)
    ax[0].set_xticklabels(classes, rotation=45, ha="right")
    ax[0].set_ylim(0, 1)
    ax[0].set_title("F1 Score")

    # --- Precision ---
    ax[1].bar(x - width/2, prec_base, width, label="Base", color=colors[0])
    ax[1].bar(x + width/2, prec_cbam, width, label="CBAM", color=colors[1])
    ax[1].set_xticks(x)
    ax[1].set_xticklabels(classes, rotation=45, ha="right")
    ax[1].set_ylim(0, 1)
    ax[1].set_title("Precision")

    # --- Recall ---
    ax[2].bar(x - width/2, rec_base, width, label="Base", color=colors[0])
    ax[2].bar(x + width/2, rec_cbam, width, label="CBAM", color=colors[1])
    ax[2].set_xticks(x)
    ax[2].set_xticklabels(classes, rotation=45, ha="right")
    ax[2].set_ylim(0, 1)
    ax[2].set_title("Recall")


    ax[2].legend(loc='upper right')

    sns.despine()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
 

In [ ]:
resumen_detallado = []
CLASSES = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])
class_indices = {cls: i for i, cls in enumerate(CLASSES)}

for exp in experiments:
    print(f"\n" + "="*50)
    print(f"Evaluating: {exp['name']}")
    print("="*50)

    best_model_path = RESULTS_PATH / f"{exp['name']}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found: {best_model_path.name}")
        continue

    custom_objects = {
        'SpatialPooling': SpatialPooling,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro
    }
    model = tf.keras.models.load_model(best_model_path, custom_objects=custom_objects, compile=False)

    X_list, y_list = [], []
    for cls in CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists(): continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0
            img = preprocess_for_backbone(img, exp['backbone'])
            X_list.append(img)
            y_list.append(class_indices[cls])

    if not X_list:
        print(f"Test images not found for {exp['name']}")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    # Preds
    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(preds, axis=1)

    prec_pc, rec_pc, f1_pc, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(CLASSES)), zero_division=0
    )

    resumen_detallado.append({
        'Model': exp['backbone'],     
        'Variant': 'cbam' if exp['use_cbam'] else 'base',
        'Precision_per_class': prec_pc,
        'Recall_per_class': rec_pc,
        'F1_per_class': f1_pc,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1_Weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    })

    # confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)))
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel("Predicted")
    plt.ylabel("Real")
    plt.title(f"Confusion Matrix: {exp['name']}")
    out_path = CONF_MATRIX_DIR / f"cm_{exp['name']}.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()

df = pd.DataFrame(resumen_detallado)

print(df[["Model", "Variant", "Accuracy", "F1_Weighted"]])

for model_name in df["Model"].unique():
    print(f"\n==================\nModel: {model_name}\n==================\n")
    plot_model_metrics(df, model_name, CLASSES,save_path=FIGS_DIRECTORY / f"{model_name}_metrics.png")

# Evaluating YOLO models upon known domain (public test)

In [ ]:
DATASET_TEST = Path("..data/dataset/dataset_YOLO/test")
PROJECT_DIR_PATH = Path("..data/models/results")

In [ ]:
MODEL_VARIANTS = ["n", "m", "l"]
RESULTS_SUMMARY = [] 

class_dirs = sorted([d.name for d in DATASET_TEST.iterdir() if d.is_dir()])
file_list = []
y_true = []

for idx, name in enumerate(class_dirs):
    images = [str(f) for f in (DATASET_TEST / name).rglob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    file_list.extend(images)
    y_true.extend([idx] * len(images))

print(f"Data is ready: {len(file_list)} images, {len(class_dirs)} classes.")

for v in MODEL_VARIANTS:
    model_path = PROJECT_DIR_PATH / f"YOLO11{v}_cls" / "weights" / "best.pt"
    
    if not model_path.exists():
        print(f"Skipping YOLO11{v}: File not found in {model_path}")
        continue

    print(f"\n🚀 Evaluating YOLO11{v}...")
    model = YOLO(model_path)
    
    start_time = time.time()
    results = model.predict(source=file_list, device=0, verbose=False)
    inference_time = (time.time() - start_time) / len(file_list) 
    
    y_pred = [r.probs.top1 for r in results]
    

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    kappa = cohen_kappa_score(y_true, y_pred)
    
    RESULTS_SUMMARY.append({
        "Model": f"YOLO11{v}",
        "Acc": acc,
        "F1-Score": f1,
        "Kappa": kappa,
        "Time/Img (s)": inference_time
    })
    

    print(f"--- Report {v.upper()} ---")
    print(classification_report(y_true, y_pred, target_names=class_dirs))

print("\n" + "="*60 + "\n📊 Model comparison\n" + "="*60)
df_results = pd.DataFrame(RESULTS_SUMMARY).set_index("Model")
print(df_results.round(4))

# EVALUATING DOMAIN GAP 

## FOR CNN-MODELS

In [ ]:
results = []

class_weights_array_for_loading = np.ones(NUM_CLASSES)

for backbone in model_list:
    print(f"\n🔹 Evaluating: {backbone}")

    model = None 
    best_model_path = RESULTS_PATH / f"{backbone}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found: {best_model_path}")
        continue 

    custom_objects = {
    "weighted_categorical_crossentropy": weighted_categorical_crossentropy,
    "recall_macro": recall_macro,
    "precision_macro": precision_macro,
    "f1_macro": f1_macro
    }  

    model = tf.keras.models.load_model(
        best_model_path,
        compile=False,
        custom_objects=custom_objects
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=weighted_categorical_crossentropy(class_weights_array_for_loading),
        metrics=[f1_macro, precision_macro, recall_macro]
    )

    # Determine the correct backbone name for the generator
    gen_backbone_name = backbone.lower().replace("_cbam", "")

    gen_public = AlbumentationsDataGenerator(TEST_PUBLIC_PATH, batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=gen_backbone_name)
    gen_local  = AlbumentationsDataGenerator(TEST_LOCAL_PATH,  batch_size=BATCH_SIZE, img_size=IMG_SIZE, backbone=gen_backbone_name)

    y_true_public = get_true_labels_from_keras_generator(gen_public)
    y_true_local  = get_true_labels_from_keras_generator(gen_local)

    y_pred_public = np.argmax(model.predict(gen_public, verbose=0), axis=1)
    y_pred_local  = np.argmax(model.predict(gen_local, verbose=0), axis=1)

    # Ensure model is not None (safety check)
    if model is None:
        print(f"Error: Model '{backbone}' could not be loaded. Skipping metrics")
        continue

    # Metrics
    report_public = classification_report(y_true_public, y_pred_public, output_dict=True, zero_division=0)
    report_local  = classification_report(y_true_local, y_pred_local, output_dict=True, zero_division=0)

    acc_public = report_public["accuracy"]
    acc_local  = report_local["accuracy"]
    f1_public  = report_public["weighted avg"]["f1-score"]
    f1_local   = report_local["weighted avg"]["f1-score"]

    domain_gap_acc = acc_public - acc_local
    domain_gap_f1  = f1_public - f1_local

    results.append({
        "Model": backbone,
        "Accuracy_Public": acc_public,
        "Accuracy_Local": acc_local,
        "F1_Public": f1_public,
        "F1_Local": f1_local,
        "Δ Accuracy": domain_gap_acc,
        "Δ F1": domain_gap_f1,
    })

    # confusion matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (y_true, y_pred, title) in zip(
        axes,
        [(y_true_public, y_pred_public, "Public Test"),
         (y_true_local, y_pred_local, "Local Test")]
    ):
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
        ax.set_title(f"{backbone} - {title}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()

# save results
df_results = pd.DataFrame(results).sort_values("Δ F1", ascending=False)
df_results.to_csv(OUTPUT_PATH / "domain_gap_metrics.csv", index=False)
display(df_results)

# Δ F1 Plot
plt.figure(figsize=(10,6))
plt.bar(df_results["Model"], df_results["Δ F1"], color="salmon")
plt.title("Domain Gap (Δ F1)")
plt.ylabel("Δ F1-score (Public - Local)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## YOLO MODELS